### Importing all packages

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import nltk
nltk.download("wordnet")
nltk.download("stopwords")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, classification_report


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\paude\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\paude\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Loading data

In [2]:
df_fake = pd.read_csv("../data/Fake.csv", sep=",")
df_true = pd.read_csv("../data/True.csv", sep=",")


In [3]:
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
df_fake.shape

(23481, 4)

In [5]:
df_fake.info()

<class 'pandas.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    23481 non-null  str  
 1   text     23481 non-null  str  
 2   subject  23481 non-null  str  
 3   date     23481 non-null  str  
dtypes: str(4)
memory usage: 60.4 MB


In [6]:
df_fake.duplicated().sum()

np.int64(3)

In [7]:
df_fake.drop_duplicates(inplace=True)

In [8]:
df_fake["Status"] = 0

In [9]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [10]:
df_true.shape

(21417, 4)

In [11]:
df_true.info()

<class 'pandas.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    21417 non-null  str  
 1   text     21417 non-null  str  
 2   subject  21417 non-null  str  
 3   date     21417 non-null  str  
dtypes: str(4)
memory usage: 51.6 MB


In [12]:
df_true.duplicated().sum()

np.int64(206)

In [13]:
df_true.drop_duplicates(inplace=True)

In [14]:
df_true["Status"] = 1

In [15]:
### Splitting last 10 data from true and fake for testing

df_true = df_true.iloc[:-10]
df_true_test = df_true.iloc[-10:]

df_fake = df_fake.iloc[:-10]
df_fake_test = df_fake.iloc[-10:]



In [16]:
df_true.shape, df_fake.shape, df_fake_test.shape, df_true_test.shape

((21201, 5), (23468, 5), (10, 5), (10, 5))

### Merging fake and true news

In [17]:
df = pd.concat([df_true, df_fake], ignore_index=True)

In [18]:
df.head()

,title,text,subject,date,Status
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [19]:
df[df["text"] == " "]

,title,text,subject,date,Status
8924,Graphic: Supreme Court roundup,,politicsNews,"June 16, 2016",1
32123,TAKE OUR POLL: Who Do You Think President Trum...,,politics,"May 10, 2017",0
32241,Joe Scarborough BERATES Mika Brzezinski Over “...,,politics,"Apr 26, 2017",0
32390,WATCH TUCKER CARLSON Scorch Sanctuary City May...,,politics,"Apr 6, 2017",0
32425,MAYOR OF SANCTUARY CITY: Trump Trying To Make ...,,politics,"Apr 2, 2017",0
...,...,...,...,...,...
43014,BALTIMORE BURNS: MARYLAND GOVERNOR BRINGS IN N...,,left-news,"Apr 27, 2015",0
43024,FULL VIDEO: THE BLOCKBUSTER INVESTIGATION INTO...,,left-news,"Apr 25, 2015",0
43025,(VIDEO) HILLARY CLINTON: RELIGIOUS BELIEFS MUS...,,left-news,"Apr 25, 2015",0
43055,(VIDEO)ICE PROTECTING OBAMA: WON’T RELEASE NAM...,,left-news,"Apr 14, 2015",0


In [20]:
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

In [21]:
df["text"].iloc[36268]

'I VE HAD IT! '

In [22]:
import string
from string import punctuation
import re

le = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def transform(text):

    # Step 1: Remove (Reuters) - tags
    text = re.sub(r'\(.*?\)\s*-', '', text)

    # Step 2: Lowercase
    text = text.lower()

    # Step 3: Replace punctuation with space (fixes SEATTLE/WASHINGTON bug)
    text = re.sub(r'[^\w\s]', ' ', text)

    # Step 4: Split into tokens
    tokens = text.split()

    # Step 5: Lemmatize and remove stopwords
    lemmatize = [le.lemmatize(word, pos="v") for word in tokens if word not in stop_words]

    return lemmatize


In [23]:
df["lem_transformed_text"] = df["text"].apply(transform)

In [24]:
df["transformed_text"] = df["lem_transformed_text"].apply(lambda x: " ".join(x))

In [25]:
df.head()

,title,text,subject,date,Status,lem_transformed_text,transformed_text
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,"[washington, head, conservative, republican, f...",washington head conservative republican factio...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,"[washington, transgender, people, allow, first...",washington transgender people allow first time...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,"[washington, special, counsel, investigation, ...",washington special counsel investigation link ...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1,"[washington, trump, campaign, adviser, george,...",washington trump campaign adviser george papad...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1,"[seattle, washington, president, donald, trump...",seattle washington president donald trump call...


In [26]:
df["original_text_char"] = df["text"].apply(lambda x: len(x))
df["lem_transformed_text_char"] = df["lem_transformed_text"].apply(lambda x: len(" " .join(x)))
df["transformed_text_char"] = df["transformed_text"].apply(lambda x: len(x))

In [27]:
df.head()

,title,text,subject,date,Status,lem_transformed_text,transformed_text,original_text_char,lem_transformed_text_char,transformed_text_char
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,"[washington, head, conservative, republican, f...",washington head conservative republican factio...,4659,3214,3214
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,"[washington, transgender, people, allow, first...",washington transgender people allow first time...,4077,2904,2904
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,"[washington, special, counsel, investigation, ...",washington special counsel investigation link ...,2789,1894,1894
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1,"[washington, trump, campaign, adviser, george,...",washington trump campaign adviser george papad...,2461,1711,1711
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1,"[seattle, washington, president, donald, trump...",seattle washington president donald trump call...,5204,3549,3549


### Selecting only required cols for training and testing

In [28]:
new_df = df[["transformed_text", "Status"]]

In [29]:
new_df.head()

,transformed_text,Status
0,washington head conservative republican factio...,1
1,washington transgender people allow first time...,1
2,washington special counsel investigation link ...,1
3,washington trump campaign adviser george papad...,1
4,seattle washington president donald trump call...,1


In [30]:
new_df.isna().sum()

transformed_text    0
Status              0
dtype: int64

In [31]:
new_df.to_csv("../data/transformed_text.csv", index=False)

In [32]:
transformed_df = pd.read_csv("../data/transformed_text.csv")

In [33]:
transformed_df.head()

,transformed_text,Status
0,washington head conservative republican factio...,1
1,washington transgender people allow first time...,1
2,washington special counsel investigation link ...,1
3,washington trump campaign adviser george papad...,1
4,seattle washington president donald trump call...,1


In [34]:
transformed_df[transformed_df.isna().any(axis=1)]

,transformed_text,Status
36268,NaN,0


In [35]:
transformed_df.dropna(inplace=True)

## Heere, i have droped those value which have only  [] as they doesnt help in predicting fake news or real news 

### Vectorization, train_test_split, and model 

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer =  TfidfVectorizer(max_features=5000, min_df=5)


In [37]:
X = vectorizer.fit_transform(new_df['transformed_text'])
y = new_df["Status"]

In [38]:
print(X.shape)

(44038, 5000)


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify= y, random_state=42
)

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

models = {
    "LR" : LogisticRegression(random_state=42),
    "MNB" : MultinomialNB(),
    "SVC" : LinearSVC(random_state=42)
}

In [41]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

scorings = ["accuracy", "precision", "f1", "recall"]

for name, model in models.items():
    row = {"Model": name}
    
    for scoring in scorings:
        scores = cross_val_score(
            model,
            X,
            y,
            cv=skf,
            scoring=scoring,
        )
        row[f"{scoring.capitalize()} Mean"] = scores.mean().round(4)
        row[f"{scoring.capitalize()} Std"]  = scores.std().round(4)
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    row["Test Accuracy"]  = round(accuracy_score(y_test, y_pred), 4)
    row["Test Precision"] = round(precision_score(y_test, y_pred), 4)
    row["Test Recall"]    = round(recall_score(y_test, y_pred), 4)
    row["Test F1"]        = round(f1_score(y_test, y_pred), 4)      
    
    results.append(row)   

In [42]:
results_df = pd.DataFrame(results)
results_df

,Model,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,F1 Mean,F1 Std,Recall Mean,Recall Std,Test Accuracy,Test Precision,Test Recall,Test F1
0,LR,0.9799,0.0014,0.9755,0.0016,0.9792,0.0015,0.9829,0.0020,0.9788,0.9732,0.9830,0.9781
1,MNB,0.9243,0.0036,0.9277,0.0039,0.9208,0.0037,0.9139,0.0041,0.9238,0.9249,0.9160,0.9205
2,SVC,0.9876,0.0005,0.9875,0.0012,0.9871,0.0005,0.9867,0.0004,0.9857,0.9846,0.9857,0.9851


In [44]:
import joblib

joblib.dump(models["SVC"], "best_model.pkl")

joblib.dump(vectorizer, "tfidf.pkl")

['tfidf.pkl']

In [45]:
model =joblib.load("best_model.pkl")
vectorizer = joblib.load("tfidf.pkl")

In [55]:
df_true_test.iloc[5]["text"]

'BUENOS AIRES (Reuters) - Argentina s main labor unions took to the streets of the capital on Tuesday demanding more jobs and protesting center-right President Mauricio Macri s economic policies.  Tens of thousands of workers gathered in the historic Plaza de Mayo criticizing Macri, who is trying to lower labor costs to attract investment and jump-start an economy that emerged from recession in the second half of last year.  If some retrograde (in the government) thinks that lowering wages, precarious living conditions and destroying trade unions is going to line up investments... we say that is very wrong,  said Juan Carlos Schmid, a leader of Argentina s largest umbrella union, the CGT. Standing on a podium at the protest, he said the CGT would meet in late September to discuss a potential strike.  Macri told Reuters in an interview this month his government was negotiating labor agreements sector by sector rather than trying to pass a comprehensive labor reform like the one approved

In [62]:
df_fake_test.iloc[9]["text"]

'RTOne of the most visible members of the armed militia that took over a wildlife refuge in Oregon says his four foster sons were taken away due to his involvement in the standoff, and he blames the federal government who  must have gotten to the governor. Robert  LaVoy  Finicum and his wife Jeanette have fostered more than 50 boys over the last decade at their ranch near Chino Valley, Arizona. The couple is licensed and has a care contract with the Catholic Charities Community Services. Many of the children came from mental hospitals, drug rehabs and group homes for emotionally distressed youth, he told Oregon Public Broadcasting (OPB). IMAGE: Robert  LaVoy  Finicum. My ranch has been a great tool for these boys,  Finicum said.  It has done a lot of good. He traveled to Oregon to take part in the takeover of the Malheur National Wildlife Refuge at the beginning of January, leaving Jeanette to care for the four boys. But now the Finicums have no more fosters to care for.A social worker

In [75]:
text = ['RTOne of the most visible members of the armed militia that took over a wildlife refuge in Oregon says his four foster sons were taken away due to his involvement in the standoff, and he blames the federal government who  must have gotten to the governor. Robert  LaVoy  Finicum and his wife Jeanette have fostered more than 50 boys over the last decade at their ranch near Chino Valley, Arizona. The couple is licensed and has a care contract with the Catholic Charities Community Services. Many of the children came from mental hospitals, drug rehabs and group homes for emotionally distressed youth, he told Oregon Public Broadcasting (OPB). IMAGE: Robert  LaVoy  Finicum. My ranch has been a great tool for these boys,  Finicum said.  It has done a lot of good. He traveled to Oregon to take part in the takeover of the Malheur National Wildlife Refuge at the beginning of January, leaving Jeanette to care for the four boys. But now the Finicums have no more fosters to care for.A social worker began removing the last four of the family s foster kids on January 4, the fourth day of the Oregon standoff. The last left five days later, he said. They didn t go out at the same time,  Finicum said.  One was there for a year, one of the boys was there six months, another eight months, and a month. I don t know where they ended up. He blamed the kids  removals on pressure from the feds. They were ripped from my wife,  Finicum said.  We are very successful [foster parents]. Our track records are good, it s been a good relationship. [Federal authorities] must have gotten to the governor, who told the state to get them out of there. Continue this story at RTREAD MORE HAMMOND RANCH NEWS AT: 21st Century Wire Hammond Ranch Files']

# Transform using the saved vectorizer
X_new = vectorizer.transform(text)

# Predict
prediction = model.predict(X_new)

if len(text[0].strip()) < 5:
    print("No proper news to detect. Please enter at least 5 characters.")
    
else:
    if prediction == 0:

        print("Fake news")
        
    else:
        print("Real news")

Fake news


In [80]:
df_true_test.iloc[1]["text"]

'BERLIN (Reuters) - The leader of Germany s Social Democrats (SPD) pledged to have U.S. nuclear weapons withdrawn from German territory if, against the odds, he defeats Angela Merkel to become chancellor next month. Addressing a campaign rally in Trier late on Tuesday, SPD leader Martin Schulz also said he, unlike Merkel, would resist demands by U.S. President Donald Trump for NATO members to increase their defense spending.  Trump wants nuclear armament. We are against this,  Schulz said, apparently trying to differentiate his party from Merkel s more hawkish Christian Democratic Union (CDU).  As chancellor, I will commit Germany to having the nuclear weapons stationed here withdrawn from our country,  he said. About 20 U.S. nuclear warheads are thought to be stationed at a military base in Buechel, in western Germany, according to unofficial estimates. The U.S. embassy in Berlin said it does not comment on nuclear weapons in Germany. Taking advantage of Trump s extreme unpopularity i